# Team GORDOBOB
Austin Jia (adj2484)\
Gordon Lee (gl23578)\
Bill Ma (bm39846)\
David Zhang (dz)

# Setup

In [3]:
# Imports
from preprocessing import clean_data, engineer_data
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor, VotingRegressor

In [ ]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/predictions.csv"

########## DEBUG ##########
DEBUG = True

########## MODEL ##########
INNER_CV = 5
OUTER_CV = 5
SCORING = 'neg_root_mean_squared_error'

# KNN (tuned in ./experiments/pca_knn.ipynb)
PCA = PCA (n_components = 29)
KNN_REGRESSOR = KNeighborsRegressor (n_neighbors = 200)

# RF (tuned in ./experiments/random_forest.ipynb)
RF_REGRESSOR = RandomForestRegressor (max_features = 'sqrt',
                                      random_state = 0,
                                      max_depth = 10,
                                      min_samples_leaf = 2,
                                      min_samples_split = 5,
                                      n_estimators = 100)

# Voting
KNN_WEIGHT = (1 / (KNN_RMSE := 5.16)) ** 3
RF_WEIGHT = (1 / (RF_RMSE := 4.49)) ** 3


# Init & Pre-Processing

In [5]:
data = pd.read_csv (TRAIN_PATH)
labels, cleaned_data = clean_data (data)
feature_engineer = FunctionTransformer (engineer_data)

# Model

In [6]:
# Build Pipelines
knn_pipeline = Pipeline ([('f_eng', feature_engineer),
                          ('scaler', StandardScaler ()),
                          ('pca', PCA),
                          ('knn', KNN_REGRESSOR)])

rf_pipeline = Pipeline ([('f_eng', feature_engineer),
                         ('pca', PCA),
                         ('knn', KNN_REGRESSOR)])

# Vote
voting_regressor = VotingRegressor ([('rf', rf_pipeline),
                                     ('knn', knn_pipeline)],
                                      weights = [RF_WEIGHT, KNN_WEIGHT])

nested_scores = cross_val_score (voting_regressor,
                                 cleaned_data,
                                 labels,
                                 cv = OUTER_CV,
                                 scoring = SCORING,
                                 n_jobs = -1)

# Evaluate
print (f"Nested RMSE: {-nested_scores.mean ()}")
if (DEBUG):
    print (f"Fold RMSEs: {-nested_scores}")

Nested RMSE: 4.79047898007782
Fold RMSEs: [4.77850768 4.79266255 4.81822813 4.78972438 4.77327215]


# Predict

In [7]:
# Build final model with all training data
final_model = voting_regressor.fit (cleaned_data, labels)

# Predict
test_data = pd.read_csv (TEST_PATH)
_, cleaned_test_data = clean_data (test_data)
predictions = final_model.predict (cleaned_test_data)

# Save
out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (predictions) + 1),
                          'Milk_Yield_L': predictions})
out_data.to_csv (OUT_PATH, index = False)